# Additional Baselines: WAIC

In the following we show how to reproduce the WAIC ensemble results.

We use the CIFAR-10 --> SVHN and CIFAR-10 --> CelebA setups as examples.

In [ ]:
from collections import defaultdict

import numpy as np
import pandas as pd

from sitn.metrics import bootstrap_auroc
from sitn.utils import construct_results_path, submit_eval_jobs, submit_train_jobs

## Model Training and Likelihood Evalutions

For WAIC, we need to train and evaluate an ensemble of five models for each dataset. Here we demonstrate how to reproduce the results using the CIFAR-10-trained ensemble.

In [ ]:
train_cfgs = []
eval_cfgs_id = []
eval_cfgs_train = []
eval_cfgs_val = []
eval_cfgs_svhn = []
eval_cfgs_celeba = []

for seed in [42, 43, 44, 45, 46]:
    train_cfg = {"dataset_name": "cifar10", "seed": seed}
    train_cfgs.append(train_cfg)

    eval_cfgs_id.append({"config": train_cfg, "split_pick": "test"})
    eval_cfgs_train.append({"config": train_cfg, "split_pick": "train"})
    eval_cfgs_val.append({"config": train_cfg, "split_pick": "val"})

    eval_cfgs_svhn.append({"config": train_cfg, "eval_dataset_name": "svhn", "split_pick": "test"})
    eval_cfgs_celeba.append({"config": train_cfg, "eval_dataset_name": "celeba", "split_pick": "test"})

In [ ]:
# Submit slurm jobs for training (or use CLI instead)
submit_train_jobs(train_cfgs)

In [ ]:
# Submit slurm jobs for evaluations (or use CLI instead)
# These jobs should only be submitted after training has completed.
# Note: at the end of training, evaluations are automatically run
# on the train, val, and test splits of the training dataset, so we
# only need to submit evaluation jobs for the OOD datasets.
submit_eval_jobs(eval_cfgs_svhn + eval_cfgs_celeba)

## Evaluate OOD Detection Performance

In [ ]:
# Collect predictions across ensemble members (seeds)
all_preds_dict = defaultdict(list)
dataset_name = train_cfgs[0]["dataset_name"]
for eval_cfg_id, eval_cfg_svhn, eval_cfg_celeba in zip(eval_cfgs_id, eval_cfgs_svhn, eval_cfgs_celeba, strict=True):
    seed = eval_cfg_id["config"]["seed"]

    # Load ID preds
    baseline_preds = pd.read_csv(construct_results_path(**eval_cfg_id, result_type="predictions"))
    baseline_preds["train_dataset"] = dataset_name
    baseline_preds["eval_dataset"] = dataset_name

    for eval_cfg_ood in [eval_cfg_svhn, eval_cfg_celeba]:
        eval_dataset_name = eval_cfg_ood["eval_dataset_name"]
        ood_preds = pd.read_csv(construct_results_path(**eval_cfg_ood, result_type="predictions"))
        ood_preds["train_dataset"] = dataset_name
        ood_preds["eval_dataset"] = eval_dataset_name

        # Combine ID and OOD preds
        preds = pd.concat([baseline_preds.copy(), ood_preds], ignore_index=True)
        preds["seed"] = seed

        all_preds_dict[eval_dataset_name].append(preds)

# Compute WAIC ensemble results
results = []
for ood_dataset, preds_list in all_preds_dict.items():
    # Ground truth targets are the same across seeds
    y_true = (preds_list[0]["eval_dataset"] != preds_list[0]["train_dataset"]).astype(int)

    # Stack scores for all seeds: shape (n_samples, n_seeds)
    seed_scores = np.stack([p["log_likelihood"].values for p in preds_list], axis=1)

    # Negative WAIC score (more OOD)
    scores = -(np.mean(seed_scores, axis=1) - np.var(seed_scores, axis=1))

    auroc, ci_lo, ci_hi = bootstrap_auroc(y_true, scores)
    results.append(
        {
            "ood_dataset": ood_dataset,
            "metric": "WAIC",
            "AUROC": auroc,
            "CI_lo": ci_lo,
            "CI_hi": ci_hi,
        }
    )

results = pd.DataFrame(results).set_index(["ood_dataset", "metric"])
results

,,AUROC,CI_lo,CI_hi
ood_dataset,metric,,,
svhn,WAIC,0.331846,0.325891,0.337807
celeba,WAIC,0.450497,0.443514,0.457555
